# Extended Analysis: Answering Follow-Up Questions

This notebook re-runs the combined poets+Quran clustering fresh and
self-contained (no dependency on files from earlier sessions), then
answers six specific follow-up questions:

1. Which surah was the outlier in Experiment 2?
2. What percentage of the Quran (by TEXT SIZE, not just surah count) fell
   into surah-only vs. mixed clusters?
3. Full table: every cluster's surah numbers/names and poet names, both
   experiments.
4. In a fresh poets-only clustering (matching the original paper's
   method, no Quran involved), which cluster(s) do the 19 poets from
   Experiment 1 fall into?
5. Experiment 1 rerun, excluding those same 19 poets.
6. Experiment 2 rerun, restricted to poets whose total surviving text is
   a similar size to a single surah.

**Embeddings are cached to disk after the first run** (`embed_cache/`),
so if you re-run this notebook later, it skips the slow embedding step
and jumps straight to clustering.

**Before you start:** put `poems.db` in the same folder as this
notebook.

Run cells top to bottom, **Shift+Enter**. First run: ~30-40 min (mostly
embedding). Later runs: a few minutes, since embeddings are cached.

In [1]:
# CELL 1 -- Install packages
!pip -q install sentence-transformers torch scikit-learn umap-learn hdbscan pandas numpy matplotlib seaborn scipy requests


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# CELL 2 -- Configuration (same values as before, plus embedding cache paths)
import re, sqlite3, hashlib, json, warnings, pickle
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

warnings.filterwarnings("ignore")

DB_PATH = Path("poems.db")
CACHE_DIR = Path("quran_cache"); CACHE_DIR.mkdir(exist_ok=True)
EMBED_CACHE_DIR = Path("embed_cache"); EMBED_CACHE_DIR.mkdir(exist_ok=True)
FIGURES_DIR = Path("output/figures"); FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR = Path("output/tables"); TABLES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR = Path("output/reports"); REPORTS_DIR.mkdir(parents=True, exist_ok=True)

SBERT_MODEL_NAME = "akhooli/Arabic-SBERT-100K"
MAX_VERSES_PER_POEM = 20

UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST = 0.1
UMAP_N_COMPONENTS_HIGH = 50
UMAP_METRIC = "cosine"
HDBSCAN_MIN_CLUSTER_SIZE = 5
HDBSCAN_MIN_SAMPLES = 3

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
import torch
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    print("GPU available:", torch.cuda.get_device_name(0))
else:
    print("No GPU found -- will run on CPU (slower).")

def split_verses(poem_text):
    if not poem_text or not isinstance(poem_text, str):
        return []
    verses = re.split(r'[\n\r]+|[.!\u061F?\u061B;]+', poem_text)
    return [v.strip() for v in verses if len(v.strip()) > 10]

plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 300, "savefig.bbox": "tight", "font.size": 11})
print("Config loaded.")

GPU available: NVIDIA GeForce RTX 4080 SUPER
Config loaded.


In [3]:
# CELL 3 -- Load the 260 poets from poems.db
conn = sqlite3.connect(str(DB_PATH))
df = pd.read_sql_query(
    "SELECT poet_name, poem_title, poem_text, poem_type, poem_meter, verses_count "
    "FROM poems WHERE poet_name IS NOT NULL AND poem_text IS NOT NULL "
    "AND LENGTH(poem_text) >= 50",
    conn,
)
conn.close()

df["poem_hash"] = df["poem_text"].apply(lambda t: hashlib.md5(t.strip().encode("utf-8")).hexdigest())
df = df.drop_duplicates(subset=["poem_hash"]).copy()

poems_by_poet = defaultdict(list)
for _, row in df.iterrows():
    poems_by_poet[row["poet_name"]].append(row["poem_text"])
poems_by_poet = dict(poems_by_poet)

# Total verse count per poet (used later for Question 6)
poet_total_verses = {
    poet: sum(len(split_verses(p)) for p in poems)
    for poet, poems in poems_by_poet.items()
}

print(f"Poets loaded: {len(poems_by_poet)}")
print(f"Poems loaded: {len(df)}")

Poets loaded: 260
Poems loaded: 2328


In [4]:
# CELL 4 -- Fetch Quran text + per-surah size metadata
cache_file = CACHE_DIR / "quran_ayat.json"

if cache_file.exists():
    print("Loading Quran from local cache...")
    with open(cache_file, encoding="utf-8") as f:
        surahs_raw = json.load(f)
else:
    print("Fetching Quran from Al Quran Cloud API...")
    resp = requests.get("https://api.alquran.cloud/v1/quran/quran-uthmani", timeout=60)
    resp.raise_for_status()
    surahs_raw = resp.json()["data"]["surahs"]
    with open(cache_file, "w", encoding="utf-8") as f:
        json.dump(surahs_raw, f, ensure_ascii=False)
    print("Fetched and cached.")

surah_names = {}
surah_poem_text = {}
surah_word_count = {}
for s in surahs_raw:
    snum = s["number"]
    surah_names[snum] = s["englishName"]
    ayat_texts = [a["text"] for a in s["ayahs"]]
    surah_poem_text[snum] = "\n".join(ayat_texts)
    surah_word_count[snum] = sum(len(a.split()) for a in ayat_texts)

# Verse count per surah (via the same split_verses function used everywhere,
# so it's directly comparable to poet_total_verses)
surah_verse_count = {snum: len(split_verses(text)) for snum, text in surah_poem_text.items()}

print(f"Surahs loaded: {len(surah_poem_text)}")
print(f"Surah verse-count range: {min(surah_verse_count.values())} to {max(surah_verse_count.values())}")
print(f"Total ayat: {sum(surah_verse_count.values())} (should be ~6236)")

Fetching Quran from Al Quran Cloud API...
Fetched and cached.
Surahs loaded: 114
Surah verse-count range: 3 to 286
Total ayat: 6235 (should be ~6236)


In [5]:
# CELL 5 -- Embed the 260 poets (cached after first run)
from sentence_transformers import SentenceTransformer

poet_cache_file = EMBED_CACHE_DIR / "poet_embeddings.pkl"

print("Loading model:", SBERT_MODEL_NAME)
model = SentenceTransformer(SBERT_MODEL_NAME)
print("Loaded. Embedding dim:", model.get_sentence_embedding_dimension())

def _encode(texts, batch_size=64):
    if not texts:
        return np.array([])
    return model.encode(texts, batch_size=batch_size, show_progress_bar=False,
                         normalize_embeddings=True, convert_to_numpy=True)

def embed_poem_verse_average(poems_by_author, max_verses=MAX_VERSES_PER_POEM, seed=RANDOM_SEED):
    out = {}
    rng = np.random.RandomState(seed)
    for author, poems in poems_by_author.items():
        poem_vectors = []
        for poem in poems:
            verses = split_verses(poem)
            if len(verses) > max_verses:
                idx = rng.choice(len(verses), max_verses, replace=False)
                verses = [verses[i] for i in sorted(idx)]
            if verses:
                poem_vectors.append(np.mean(_encode(verses), axis=0))
        if poem_vectors:
            out[author] = np.mean(poem_vectors, axis=0)
    return out

if poet_cache_file.exists():
    print("Loading cached poet embeddings...")
    with open(poet_cache_file, "rb") as f:
        poet_embeddings = pickle.load(f)
    print(f"Loaded {len(poet_embeddings)} cached poet embeddings.")
else:
    print("Embedding 260 poets (this is the slow step, one-time only)...")
    poet_embeddings = embed_poem_verse_average(poems_by_poet)
    with open(poet_cache_file, "wb") as f:
        pickle.dump(poet_embeddings, f)
    print(f"Done and cached. {len(poet_embeddings)} poets embedded.")

Loading model: akhooli/Arabic-SBERT-100K


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded. Embedding dim: 768
Embedding 260 poets (this is the slow step, one-time only)...
Done and cached. 260 poets embedded.


In [6]:
# CELL 6 -- Embed the 114 surahs (cached after first run)
surah_cache_file = EMBED_CACHE_DIR / "surah_vectors.pkl"

if surah_cache_file.exists():
    print("Loading cached surah embeddings...")
    with open(surah_cache_file, "rb") as f:
        surah_vectors = pickle.load(f)
    print(f"Loaded {len(surah_vectors)} cached surah embeddings.")
else:
    print("Embedding 114 surahs...")
    surah_vectors = {}
    for snum, poem_text in surah_poem_text.items():
        verses = split_verses(poem_text)
        if verses:
            surah_vectors[snum] = np.mean(_encode(verses), axis=0)
    with open(surah_cache_file, "wb") as f:
        pickle.dump(surah_vectors, f)
    print(f"Done and cached. {len(surah_vectors)} surahs embedded.")

quran_whole_vector = np.mean(list(surah_vectors.values()), axis=0)
print("Combined whole-Quran vector computed.")

Embedding 114 surahs...
Done and cached. 114 surahs embedded.
Combined whole-Quran vector computed.


In [7]:
# CELL 7 -- Shared clustering function, then re-run Experiments 1 and 2
import umap, hdbscan
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import silhouette_score

def run_umap(matrix, n_components, n_neighbors, min_dist, metric="cosine", random_state=RANDOM_SEED):
    reducer = umap.UMAP(n_neighbors=min(n_neighbors, len(matrix) - 1), n_components=n_components,
                         min_dist=min_dist, metric=metric, random_state=random_state)
    return reducer.fit_transform(matrix)

def cluster_and_report(embeddings_dict, label_types, title, n_neighbors=UMAP_N_NEIGHBORS,
                       min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE, verbose=True):
    names = sorted(embeddings_dict.keys())
    n = len(names)
    emb_matrix = np.array([embeddings_dict[nm] for nm in names])

    umap_high = run_umap(emb_matrix, min(UMAP_N_COMPONENTS_HIGH, n - 1), n_neighbors, 0.0, UMAP_METRIC)
    clusterer = hdbscan.HDBSCAN(min_cluster_size=min(min_cluster_size, max(2, n // 10)),
                                min_samples=HDBSCAN_MIN_SAMPLES,
                                cluster_selection_method="eom", metric="euclidean")
    labels = clusterer.fit_predict(umap_high)
    umap_2d = run_umap(emb_matrix, 2, n_neighbors, UMAP_MIN_DIST, UMAP_METRIC)

    mask = labels >= 0
    n_clusters = len(set(labels[mask])) if mask.sum() > 0 else 0
    n_outliers = int(np.sum(labels == -1))
    sil = float(silhouette_score(umap_high[mask], labels[mask])) if n_clusters >= 2 and mask.sum() > n_clusters else None

    result_df = pd.DataFrame({
        "name": names, "type": [label_types[nm] for nm in names],
        "cluster": labels, "umap_x": umap_2d[:, 0], "umap_y": umap_2d[:, 1],
    })
    if verbose:
        print(f"=== {title} ===")
        print(f"Total: {n} | Clusters: {n_clusters} | Outliers: {n_outliers} | Silhouette: {sil}")
    return result_df, {"n_entities": n, "n_clusters": n_clusters, "n_outliers": n_outliers, "silhouette": sil}, (names, cosine_similarity(emb_matrix))

# Re-run Experiment 1: 260 poets + Quran (whole) = 261
exp1_embeddings = dict(poet_embeddings)
exp1_embeddings["Quran (whole)"] = quran_whole_vector
exp1_types = {nm: ("Quran" if nm == "Quran (whole)" else "Poet") for nm in exp1_embeddings}
exp1_df, exp1_metrics, (exp1_names, exp1_sim) = cluster_and_report(
    exp1_embeddings, exp1_types, "Experiment 1: 260 poets + Quran (whole) = 261")

quran_row = exp1_df[exp1_df["name"] == "Quran (whole)"].iloc[0]
the_19_poets = exp1_df[(exp1_df["cluster"] == quran_row["cluster"]) & (exp1_df["name"] != "Quran (whole)")]["name"].tolist()
print(f"\nQuran's cluster: {quran_row['cluster']} | Poets sharing it: {len(the_19_poets)}")

# Re-run Experiment 2: 260 poets + 114 surahs = 374
exp2_embeddings = dict(poet_embeddings)
for snum, vec in surah_vectors.items():
    exp2_embeddings[f"Surah {snum}: {surah_names[snum]}"] = vec
exp2_types = {nm: ("Surah" if nm.startswith("Surah ") else "Poet") for nm in exp2_embeddings}
exp2_df, exp2_metrics, (exp2_names, exp2_sim) = cluster_and_report(
    exp2_embeddings, exp2_types, "Experiment 2: 260 poets + 114 surahs = 374")

exp1_df.to_csv(TABLES_DIR / "experiment1_clusters.csv", index=False)
exp2_df.to_csv(TABLES_DIR / "experiment2_clusters.csv", index=False)
print("\nBoth baseline experiments re-run and saved.")

=== Experiment 1: 260 poets + Quran (whole) = 261 ===
Total: 261 | Clusters: 13 | Outliers: 40 | Silhouette: 0.46884259581565857

Quran's cluster: 1 | Poets sharing it: 19
=== Experiment 2: 260 poets + 114 surahs = 374 ===
Total: 374 | Clusters: 12 | Outliers: 11 | Silhouette: 0.46887871623039246

Both baseline experiments re-run and saved.


## Question 1 — Which surah was the outlier in Experiment 2?

In [8]:
# CELL 8 -- Q1: which surah(s) were outliers in Experiment 2?
outlier_surahs = exp2_df[(exp2_df["type"] == "Surah") & (exp2_df["cluster"] == -1)]
print(f"Surahs classified as outliers: {len(outlier_surahs)}")
print(outlier_surahs[["name"]].to_string(index=False))

Surahs classified as outliers: 1
               name
Surah 111: Al-Masad


## Question 2 — Surah-count percentage vs. text-size percentage

The 74% figure given earlier was by COUNT (84 of 114 surahs). This
computes the same split weighted by actual TEXT SIZE (total ayat), since
a cluster full of short surahs vs. long surahs would look very different
by that measure.

In [9]:
# CELL 9 -- Q2: percentage by surah count vs. percentage by text size
exp2_df["surah_number"] = exp2_df["name"].apply(
    lambda n: int(n.split(":")[0].replace("Surah ", "")) if n.startswith("Surah ") else None)
exp2_df["verse_count"] = exp2_df["surah_number"].map(surah_verse_count)

cluster_composition = exp2_df.groupby("cluster")["type"].value_counts().unstack(fill_value=0)
surah_only_clusters = cluster_composition[(cluster_composition.get("Poet", 0) == 0) & (cluster_composition.index != -1)].index.tolist()
mixed_clusters = cluster_composition[(cluster_composition.get("Poet", 0) > 0) & (cluster_composition.get("Surah", 0) > 0)].index.tolist()

surah_rows = exp2_df[exp2_df["type"] == "Surah"].copy()
total_ayat = surah_rows["verse_count"].sum()

group_labels = []
for _, row in surah_rows.iterrows():
    if row["cluster"] == -1:
        group_labels.append("Outlier")
    elif row["cluster"] in surah_only_clusters:
        group_labels.append("Surah-only cluster")
    elif row["cluster"] in mixed_clusters:
        group_labels.append("Mixed cluster (with poets)")
    else:
        group_labels.append("Other")
surah_rows["group"] = group_labels

summary = surah_rows.groupby("group").agg(
    n_surahs=("name", "count"), total_ayat=("verse_count", "sum")
)
summary["pct_by_surah_count"] = (summary["n_surahs"] / len(surah_rows) * 100).round(1)
summary["pct_by_ayat_count"] = (summary["total_ayat"] / total_ayat * 100).round(1)

print(f"Total surahs: {len(surah_rows)} | Total ayat across all surahs: {total_ayat}\n")
print(summary[["n_surahs", "pct_by_surah_count", "total_ayat", "pct_by_ayat_count"]].to_string())

Total surahs: 114 | Total ayat across all surahs: 6235.0

                            n_surahs  pct_by_surah_count  total_ayat  pct_by_ayat_count
group                                                                                  
Mixed cluster (with poets)        29                25.4       841.0               13.5
Outlier                            1                 0.9         5.0                0.1
Surah-only cluster                84                73.7      5389.0               86.4


## Question 3 — Full cluster membership table (both experiments)

Every cluster, listing which surah numbers/names and which poets fall
into it.

In [10]:
# CELL 10 -- Q3: full cluster membership table, both experiments
def build_cluster_table(df, source_label):
    rows = []
    for cl in sorted(df["cluster"].unique()):
        members = df[df["cluster"] == cl]
        poets = members[members["type"] == "Poet"]["name"].tolist()
        surahs = members[members["type"].isin(["Surah", "Quran"])]["name"].tolist()
        rows.append({
            "experiment": source_label,
            "cluster": "Outliers" if cl == -1 else f"Cluster {cl}",
            "n_poets": len(poets), "n_surahs": len(surahs),
            "poet_names": "; ".join(poets),
            "surah_names": "; ".join(surahs),
        })
    return pd.DataFrame(rows)

exp1_table = build_cluster_table(exp1_df, "Experiment 1")
exp2_table = build_cluster_table(exp2_df, "Experiment 2")
full_table = pd.concat([exp1_table, exp2_table], ignore_index=True)
full_table.to_csv(TABLES_DIR / "full_cluster_membership.csv", index=False)

print(f"Saved to {TABLES_DIR / 'full_cluster_membership.csv'}")
print(f"({len(full_table)} rows total -- open the CSV for the full poet/surah name lists,")
print(" they're long and don't display cleanly in a notebook cell)")
full_table[["experiment", "cluster", "n_poets", "n_surahs"]]

Saved to output\tables\full_cluster_membership.csv
(27 rows total -- open the CSV for the full poet/surah name lists,
 they're long and don't display cleanly in a notebook cell)


,experiment,cluster,n_poets,n_surahs
0,Experiment 1,Outliers,40,0
1,Experiment 1,Cluster 0,5,0
2,Experiment 1,Cluster 1,19,1
3,Experiment 1,Cluster 2,12,0
4,Experiment 1,Cluster 3,28,0
5,Experiment 1,Cluster 4,6,0
6,Experiment 1,Cluster 5,10,0
7,Experiment 1,Cluster 6,8,0
8,Experiment 1,Cluster 7,11,0
9,Experiment 1,Cluster 8,68,0


## Question 4 — Where do the 19 poets land in a poets-only clustering?

Re-clusters just the 260 poets (no Quran at all), using the exact same
method/parameters as the original paper reproduction, then looks up
which cluster each of the 19 poets from Experiment 1 falls into there.
This tells you whether those 19 poets already grouped together in the
paper's own clustering, or were previously scattered and only came
together once the Quran was added.

In [11]:
# CELL 11 -- Q4: poets-only clustering (matching the paper), cross-referenced
poets_only_types = {nm: "Poet" for nm in poet_embeddings}
poets_only_df, poets_only_metrics, _ = cluster_and_report(
    poet_embeddings, poets_only_types, "Poets-only clustering (paper method, no Quran)")

lookup = dict(zip(poets_only_df["name"], poets_only_df["cluster"]))
q4_rows = [{"poet": p, "cluster_without_quran": lookup.get(p, "N/A")} for p in the_19_poets]
q4_df = pd.DataFrame(q4_rows).sort_values("cluster_without_quran")
q4_df.to_csv(TABLES_DIR / "question4_poets_without_quran.csv", index=False)

print(f"\nPoets-only clustering found {poets_only_metrics['n_clusters']} clusters "
      f"(paper reports 12 -- differences are normal run-to-run variation).\n")
print("Where the 19 poets land WITHOUT the Quran in the mix:")
print(q4_df.to_string(index=False))
print(f"\nDistinct clusters these 19 poets are spread across: {q4_df['cluster_without_quran'].nunique()}")

=== Poets-only clustering (paper method, no Quran) ===
Total: 260 | Clusters: 13 | Outliers: 37 | Silhouette: 0.42085936665534973

Poets-only clustering found 13 clusters (paper reports 12 -- differences are normal run-to-run variation).

Where the 19 poets land WITHOUT the Quran in the mix:
              poet  cluster_without_quran
       أوس الهجيمي                     -1
 ابن شعواء الفزاري                      2
      الأعرج المري                      2
     الأشعث الجاشي                      2
  السليك بن السلكة                      2
امرؤ القيس الزهيري                      2
     المعان بن روق                      2
    المثلم الفزاري                      2
      جبار الفزاري                      2
    الهجرس التغلبي                      2
      هند الفزارية                      2
     نهيكة الفزاري                      2
      هند بنت الخس                      2
  ابن كلاب العقيلي                      3
  أبو حسان الفزاري                      3
  أبو بثينه الهذلي                 

## Question 5 — Experiment 1 rerun, excluding the 19 poets

Same as Experiment 1 (Quran as one author vs. all poets), but the 19
poets who shared a cluster with the Quran the first time are removed
before clustering. 241 poets + Quran = 242 total.

In [12]:
# CELL 12 -- Q5: Experiment 1 rerun, excluding the 19 poets
exp5_embeddings = {p: v for p, v in poet_embeddings.items() if p not in the_19_poets}
exp5_embeddings["Quran (whole)"] = quran_whole_vector
exp5_types = {nm: ("Quran" if nm == "Quran (whole)" else "Poet") for nm in exp5_embeddings}

exp5_df, exp5_metrics, (exp5_names, exp5_sim) = cluster_and_report(
    exp5_embeddings, exp5_types,
    f"Experiment 5 (Q5): {len(exp5_embeddings)-1} poets (19 excluded) + Quran (whole)")

quran_row5 = exp5_df[exp5_df["name"] == "Quran (whole)"].iloc[0]
same_cluster5 = exp5_df[(exp5_df["cluster"] == quran_row5["cluster"]) & (exp5_df["name"] != "Quran (whole)")]
print(f"\nQuran's new cluster: {quran_row5['cluster']}")
print(f"Poets now sharing it: {len(same_cluster5)}")
if len(same_cluster5) > 0:
    print(same_cluster5["name"].tolist())

qidx5 = exp5_names.index("Quran (whole)")
sims5 = exp5_sim[qidx5].copy(); sims5[qidx5] = -1
top5_idx5 = np.argsort(sims5)[-5:][::-1]
print("\nTop 5 most similar poets to the Quran now:")
for i in top5_idx5:
    print(f"  {exp5_names[i]}: {sims5[i]:.4f}")

exp5_df.to_csv(TABLES_DIR / "experiment5_clusters_excl19.csv", index=False)

=== Experiment 5 (Q5): 241 poets (19 excluded) + Quran (whole) ===
Total: 242 | Clusters: 12 | Outliers: 49 | Silhouette: 0.5480881333351135

Quran's new cluster: -1
Poets now sharing it: 48
['آمنة بنت عتيبة', 'أبو الفضل الكناني', 'أبو المثلم الهذلي', 'أروى بنت الحباب', 'أزبر بن غزي', 'أسماء التغلبية', 'أوس العبدي', 'ابن أم حزنة', 'ابن المضلَّل', 'ابن زرعة الباهلي', 'ابن مالك الحميري', 'ابنة حكيم بن عمرو العبدية', 'الأسفع الأرحبي', 'الأسلوم اليامي', 'الأضبط السعدي', 'الحارث الجرمي', 'الحارث بن مر', 'الخنساء بنت التيجان', 'السلكة أم السليك', 'القعقاع بن شبث اليهودي', 'المرار الكلبي', 'المرقش الأكبر', 'المفضل النكري', 'النابغة الغنوي', 'النوار الجل', 'الهيفاء بنت صبيح القضاعية', 'بداء بن سليمان', 'بشامة بن الغدير', 'بشر بن أبي خازم', 'بكر الجرهمي', 'بيهس الغطفاني', 'تماضر بنت الشريد', 'ثعلبة بن عامر', 'جابر المرني', 'حاتم الطائي', 'دوسر بن هذيل', 'زهير بن أبي سلمى', 'سلامة بن جندل', 'صخر الغي', 'طرفة بن العبد', 'عبد المسيح بن عسلة', 'عبد يغوث الحارثي', 'علقمة الفحل', 'عمرو بن كلثوم', 'لق

## Question 6 — Experiment 2 rerun, restricted to similarly-sized poets

Surahs range from 3 to 286 verses/ayat. This restricts the poet pool to
only poets whose TOTAL surviving output (across all their poems, in
verses) falls in that same range — i.e., poets whose entire body of work
is comparable in scale to a single surah, rather than including
prolific poets with hundreds of verses across many poems.

In [13]:
# CELL 13 -- Q6: Experiment 2 rerun, restricted to poets sized like a single surah
min_surah_verses = min(surah_verse_count.values())
max_surah_verses = max(surah_verse_count.values())
print(f"Surah size range (verses/ayat): {min_surah_verses} to {max_surah_verses}")

qualifying_poets = {p: v for p, v in poet_total_verses.items() if min_surah_verses <= v <= max_surah_verses}
print(f"Poets whose total output falls in that range: {len(qualifying_poets)} of {len(poet_total_verses)}")

exp6_embeddings = {p: poet_embeddings[p] for p in qualifying_poets if p in poet_embeddings}
for snum, vec in surah_vectors.items():
    exp6_embeddings[f"Surah {snum}: {surah_names[snum]}"] = vec
exp6_types = {nm: ("Surah" if nm.startswith("Surah ") else "Poet") for nm in exp6_embeddings}

exp6_df, exp6_metrics, _ = cluster_and_report(
    exp6_embeddings, exp6_types,
    f"Experiment 6 (Q6): {len(qualifying_poets)} size-matched poets + 114 surahs")

cluster_composition6 = exp6_df.groupby("cluster")["type"].value_counts().unstack(fill_value=0)
print("\nCluster composition (poets vs surahs per cluster):")
print(cluster_composition6)

surah_only6 = cluster_composition6[(cluster_composition6.get("Poet", 0) == 0) & (cluster_composition6.index != -1)]
mixed6 = cluster_composition6[(cluster_composition6.get("Poet", 0) > 0) & (cluster_composition6.get("Surah", 0) > 0)]
surah_rows6 = exp6_df[exp6_df["type"] == "Surah"]
surah_outliers6 = surah_rows6[surah_rows6["cluster"] == -1]

print(f"\nSurah-only clusters: {len(surah_only6)}")
print(f"Mixed clusters (poets + surahs): {len(mixed6)}")
print(f"Surahs as outliers: {len(surah_outliers6)} / {len(surah_rows6)}")

exp6_df.to_csv(TABLES_DIR / "experiment6_clusters_sizematched.csv", index=False)

Surah size range (verses/ayat): 3 to 286
Poets whose total output falls in that range: 213 of 260
=== Experiment 6 (Q6): 213 size-matched poets + 114 surahs ===
Total: 327 | Clusters: 13 | Outliers: 27 | Silhouette: 0.6067867279052734

Cluster composition (poets vs surahs per cluster):
type     Poet  Surah
cluster             
-1         27      0
 0          0     61
 1          2     17
 2          3     22
 3         25      0
 4          0      8
 5         11      6
 6         75      0
 7          8      0
 8          5      0
 9          6      0
 10         6      0
 11        19      0
 12        26      0

Surah-only clusters: 2
Mixed clusters (poets + surahs): 3
Surahs as outliers: 0 / 114


In [14]:
# CELL 14 -- Final report answering all 6 questions
report_path = REPORTS_DIR / "extended_analysis_report.txt"
with open(report_path, "w", encoding="utf-8") as f:
    f.write("=" * 70 + "\n")
    f.write("EXTENDED ANALYSIS: SIX FOLLOW-UP QUESTIONS\n")
    f.write("=" * 70 + "\n\n")

    f.write("Q1: Which surah was the outlier in Experiment 2?\n" + "-" * 50 + "\n")
    f.write(f"  {outlier_surahs['name'].tolist()}\n\n")

    f.write("Q2: Percentage by count vs. by text size\n" + "-" * 50 + "\n")
    f.write(summary[["n_surahs", "pct_by_surah_count", "total_ayat", "pct_by_ayat_count"]].to_string() + "\n\n")

    f.write("Q3: Full cluster membership table\n" + "-" * 50 + "\n")
    f.write("  See output/tables/full_cluster_membership.csv\n\n")

    f.write("Q4: Where the 19 poets land WITHOUT the Quran (paper-style clustering)\n" + "-" * 50 + "\n")
    f.write(q4_df.to_string(index=False) + "\n")
    f.write(f"  Distinct clusters: {q4_df['cluster_without_quran'].nunique()}\n\n")

    f.write("Q5: Experiment 1 rerun excluding the 19 poets\n" + "-" * 50 + "\n")
    for k, v in exp5_metrics.items():
        f.write(f"  {k}: {v}\n")
    f.write(f"  Quran's new cluster: {quran_row5['cluster']}\n")
    f.write(f"  New poets sharing it: {len(same_cluster5)}\n")
    if len(same_cluster5) > 0:
        f.write(f"    {same_cluster5['name'].tolist()}\n")
    f.write("  Top 5 nearest poets now:\n")
    for i in top5_idx5:
        f.write(f"    {exp5_names[i]}: {sims5[i]:.4f}\n")

    f.write("\nQ6: Experiment 2 restricted to size-matched poets\n" + "-" * 50 + "\n")
    f.write(f"  Surah size range: {min_surah_verses} to {max_surah_verses} verses\n")
    f.write(f"  Qualifying poets: {len(qualifying_poets)} of {len(poet_total_verses)}\n")
    for k, v in exp6_metrics.items():
        f.write(f"  {k}: {v}\n")
    f.write(f"  Surah-only clusters: {len(surah_only6)}\n")
    f.write(f"  Mixed clusters: {len(mixed6)}\n")
    f.write(f"  Surah outliers: {len(surah_outliers6)} / {len(surah_rows6)}\n")

print(f"Report written to {report_path}")
print()
print(open(report_path, encoding="utf-8").read())

Report written to output\reports\extended_analysis_report.txt

EXTENDED ANALYSIS: SIX FOLLOW-UP QUESTIONS

Q1: Which surah was the outlier in Experiment 2?
--------------------------------------------------
  ['Surah 111: Al-Masad']

Q2: Percentage by count vs. by text size
--------------------------------------------------
                            n_surahs  pct_by_surah_count  total_ayat  pct_by_ayat_count
group                                                                                  
Mixed cluster (with poets)        29                25.4       841.0               13.5
Outlier                            1                 0.9         5.0                0.1
Surah-only cluster                84                73.7      5389.0               86.4

Q3: Full cluster membership table
--------------------------------------------------
  See output/tables/full_cluster_membership.csv

Q4: Where the 19 poets land WITHOUT the Quran (paper-style clustering)
----------------------------

## Done

All six questions answered above, plus saved to `output/`:
- `output/tables/full_cluster_membership.csv` — Q3's complete table
- `output/tables/question4_poets_without_quran.csv` — Q4's cross-reference
- `output/tables/experiment5_clusters_excl19.csv` — Q5's full rerun
- `output/tables/experiment6_clusters_sizematched.csv` — Q6's full rerun
- `output/reports/extended_analysis_report.txt` — everything in one text file

**Reading Q5 and Q6:** if excluding the 19 poets (Q5) makes the Quran an
outlier or pushes its nearest-neighbor similarity down noticeably, that
strengthens the idea those 19 specifically were pulling it into a
cluster. If Q6 still shows a meaningful mixed-cluster percentage even
after restricting to only similarly-sized poets, that weakens the
"it's just a sample-size artifact" explanation from the earlier report —
worth flagging either way when you write this up.